# 03: TMDB Split Preparation

## Setup

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.eda.helper import entity_embedding_dim
from src.eda.report import log_findings, savefig

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
FIGURES_DIR = REPO_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

## Load chosen candidates

In [ ]:
tmdb_budget_revenue_gt0 = pd.read_parquet(PROCESSED_DIR / "tmdb_br_gt0.parquet")
tmdb_no_budget_revenue_gt5 = pd.read_parquet(PROCESSED_DIR / "tmdb_nbr_gt5.parquet")

print(f"tmdb_budget_revenue_gt0: {tmdb_budget_revenue_gt0.shape}")
print(f"tmdb_no_budget_revenue_gt5: {tmdb_no_budget_revenue_gt5.shape}")

## 1. Train/val/test split

Stratified splits on `original_language` for both chosen candidates: `tmdb_budget_revenue_gt0` at 80/10/10 and `tmdb_no_budget_revenue_gt5` at 70/15/15 (its larger row count affords a bigger held-out val/test share). Splitting happens now, before any further feature engineering, to avoid leakage between the eventual train/val/test sets. Integrity is checked and shapes are printed only -- nothing is saved yet.

### Stratified split by `original_language`

In [ ]:
from sklearn.model_selection import train_test_split


def stratified_three_way_split(df, stratify_col, train_size, val_size, test_size, random_state=42, min_class_size=10):
    assert abs(train_size + val_size + test_size - 1.0) < 1e-9

    # Disposable stratify key: languages below min_class_size cannot be
    # divided across 3 splits (sklearn needs >=2 members per class per
    # split), so they are pooled into "other" for stratification only.
    value_counts = df[stratify_col].value_counts()
    rare_values = value_counts[value_counts < min_class_size].index
    df = df.copy()
    df["stratify_key"] = df[stratify_col].astype(object).where(~df[stratify_col].isin(rare_values), "other")

    train_df, temp_df = train_test_split(
        df,
        train_size=train_size,
        stratify=df["stratify_key"],
        random_state=random_state,
    )

    val_relative_size = val_size / (val_size + test_size)
    val_df, test_df = train_test_split(
        temp_df,
        train_size=val_relative_size,
        stratify=temp_df["stratify_key"],
        random_state=random_state,
    )

    train_df = train_df.drop(columns="stratify_key")
    val_df = val_df.drop(columns="stratify_key")
    test_df = test_df.drop(columns="stratify_key")

    return train_df, val_df, test_df


def assert_split_integrity(original_df, train_df, val_df, test_df, id_col="id"):
    assert len(train_df) + len(val_df) + len(test_df) == len(original_df), (
        f"split sizes {len(train_df)}+{len(val_df)}+{len(test_df)} != "
        f"input {len(original_df)}"
    )
    train_ids = set(train_df[id_col])
    val_ids = set(val_df[id_col])
    test_ids = set(test_df[id_col])
    assert not (train_ids & val_ids), "id overlap between train and val"
    assert not (train_ids & test_ids), "id overlap between train and test"
    assert not (val_ids & test_ids), "id overlap between val and test"

In [ ]:
tmdb_br_gt0_train, tmdb_br_gt0_val, tmdb_br_gt0_test = stratified_three_way_split(
    tmdb_budget_revenue_gt0,
    stratify_col="original_language",
    train_size=0.8, val_size=0.1, test_size=0.1,
)
assert_split_integrity(
    tmdb_budget_revenue_gt0, tmdb_br_gt0_train, tmdb_br_gt0_val, tmdb_br_gt0_test
)

tmdb_nbr_gt5_train, tmdb_nbr_gt5_val, tmdb_nbr_gt5_test = stratified_three_way_split(
    tmdb_no_budget_revenue_gt5,
    stratify_col="original_language",
    train_size=0.7, val_size=0.15, test_size=0.15,
)
assert_split_integrity(
    tmdb_no_budget_revenue_gt5, tmdb_nbr_gt5_train, tmdb_nbr_gt5_val, tmdb_nbr_gt5_test
)

print("tmdb_budget_revenue_gt0 (80/10/10):")
print(f"  train: {tmdb_br_gt0_train.shape}")
print(f"  val:   {tmdb_br_gt0_val.shape}")
print(f"  test:  {tmdb_br_gt0_test.shape}")

print("tmdb_no_budget_revenue_gt5 (70/15/15):")
print(f"  train: {tmdb_nbr_gt5_train.shape}")
print(f"  val:   {tmdb_nbr_gt5_val.shape}")
print(f"  test:  {tmdb_nbr_gt5_test.shape}")

## 2. Language coverage curve

For each candidate's training split, checks how many `original_language` values meet a minimum-count threshold, and what fraction of training rows those languages collectively cover, for thresholds in [10, 25, 50, 100, 150, 200, 300]. Diagnostic only -- no rare-language bucketing is applied here.

In [ ]:
LANGUAGE_THRESHOLDS = [10, 25, 50, 75, 100, 150, 200, 300]
train_splits_for_language_coverage = {
    "tmdb_budget_revenue_gt0": tmdb_br_gt0_train,
    "tmdb_no_budget_revenue_gt5": tmdb_nbr_gt5_train,
}

def language_coverage_curve(df, stratify_col="original_language", thresholds=LANGUAGE_THRESHOLDS):
    counts = df[stratify_col].value_counts()
    total = len(df)
    return pd.DataFrame(
        {
            "n_languages": [int((counts >= t).sum()) for t in thresholds],
            "row_coverage": [counts[counts >= t].sum() / total for t in thresholds],
        },
        index=pd.Index(thresholds, name="threshold"),
    )

language_coverage_curves = {
    name: language_coverage_curve(df) for name, df in train_splits_for_language_coverage.items()
}

### Language coverage table

In [ ]:
language_coverage_table = pd.concat(language_coverage_curves, axis=1)
language_coverage_table.round(3)

### Coverage curves

For each candidate, plots language count and row coverage against the minimum-count threshold, side by side.

In [ ]:
# Same muted blue and near-black ink as the vocabulary coverage
# plots below, kept local since this section runs before that one
# defines them.
CURVE_COLOR = "#2a78d6"
TEXT_COLOR = "#0b0b0b"

language_coverage_metrics = [
    ("n_languages", "Languages meeting threshold"),
    ("row_coverage", "Row coverage"),
]

for candidate_name, curve in language_coverage_curves.items():
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

    for ax, (metric, ylabel) in zip(axes, language_coverage_metrics):
        ax.plot(curve.index, curve[metric], color=CURVE_COLOR, marker="o", linewidth=2)
        ax.set_title(ylabel, fontsize=13, color=TEXT_COLOR)
        ax.set_xlabel("Minimum-count threshold", fontsize=12, color=TEXT_COLOR)
        ax.set_ylabel(ylabel, fontsize=12, color=TEXT_COLOR)
        ax.tick_params(axis="both", labelsize=10, colors=TEXT_COLOR)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        if metric == "row_coverage":
            ax.set_ylim(0, 1.05)

    fig.suptitle(candidate_name, fontsize=14, color=TEXT_COLOR)
    fig.tight_layout()
    savefig(fig, f"03_{candidate_name}_language_coverage.png")
    plt.show()

## 3. Rare-language bucketing

Languages with fewer than 100 occurrences in each candidate's *training* split are pooled into `"other"`. The training split alone defines the kept-language set; val and test reuse that exact set instead of recounting their own rare languages, so no split's vocabulary choice leaks information from another split.

In [ ]:
BUCKET_MIN_COUNT = 100

# Snapshot pre-bucket training-split language shares for the
# stratification sanity check below, before "other" merges rare
# languages together.
pre_bucket_train_language_share = {
    "tmdb_budget_revenue_gt0": tmdb_br_gt0_train["original_language"].value_counts(normalize=True),
    "tmdb_no_budget_revenue_gt5": tmdb_nbr_gt5_train["original_language"].value_counts(normalize=True),
}

def bucket_rare_languages(train_df, val_df, test_df, stratify_col="original_language", min_count=BUCKET_MIN_COUNT):
    train_counts = train_df[stratify_col].value_counts()
    kept_languages = set(train_counts[train_counts >= min_count].index)

    def apply_bucket(df):
        df = df.copy()
        mask = df[stratify_col].isin(kept_languages)
        df[stratify_col] = df[stratify_col].astype(str).where(mask, "other")
        return df

    return apply_bucket(train_df), apply_bucket(val_df), apply_bucket(test_df), kept_languages


tmdb_br_gt0_train, tmdb_br_gt0_val, tmdb_br_gt0_test, br_gt0_kept_languages = bucket_rare_languages(
    tmdb_br_gt0_train, tmdb_br_gt0_val, tmdb_br_gt0_test
)
tmdb_nbr_gt5_train, tmdb_nbr_gt5_val, tmdb_nbr_gt5_test, nbr_gt5_kept_languages = bucket_rare_languages(
    tmdb_nbr_gt5_train, tmdb_nbr_gt5_val, tmdb_nbr_gt5_test
)

bucketed_splits = {
    "tmdb_budget_revenue_gt0": {"train": tmdb_br_gt0_train, "val": tmdb_br_gt0_val, "test": tmdb_br_gt0_test},
    "tmdb_no_budget_revenue_gt5": {"train": tmdb_nbr_gt5_train, "val": tmdb_nbr_gt5_val, "test": tmdb_nbr_gt5_test},
}

for candidate_name, splits in bucketed_splits.items():
    for split_name, df in splits.items():
        n_categories = df["original_language"].nunique()
        n_other = (df["original_language"] == "other").sum()
        print(f"{candidate_name} / {split_name}: {n_categories} categories, {n_other} rows in 'other'")

## 4. Recompute embedding dimension for `original_language`

`00_tmdb_eda.ipynb`'s `embedding_dim_table` sized `original_language`'s entity-embedding dimension off its raw 179-code cardinality. Section 3 above bucketed each candidate's training split down to its kept-language set plus `"other"`, so the dimension `entity_embedding_dim()` would actually suggest at train time is much smaller. Recomputed here for both candidates, printed alongside the original 179-based figure.

In [ ]:
RAW_LANGUAGE_CARDINALITY = 179
raw_embedding_dim = entity_embedding_dim(RAW_LANGUAGE_CARDINALITY)

post_binning_cardinality = {
    "tmdb_budget_revenue_gt0": len(br_gt0_kept_languages) + 1,  # +1 for "other"
    "tmdb_no_budget_revenue_gt5": len(nbr_gt5_kept_languages) + 1,
}

embedding_dim_comparison = pd.DataFrame(
    {
        "cardinality": post_binning_cardinality,
        "suggested_embedding_dim": {
            name: entity_embedding_dim(card) for name, card in post_binning_cardinality.items()
        },
    }
)
embedding_dim_comparison.index.name = "candidate"

print(f"Raw cardinality (00_tmdb_eda.ipynb): {RAW_LANGUAGE_CARDINALITY} -> suggested_embedding_dim = {raw_embedding_dim}")
embedding_dim_comparison

## 5. Top-K vocabulary coverage

For `keywords` and `production_companies` (both list-valued), checks what fraction of total item occurrences the top-K most frequent items cover, for K in [50, 100, 250, 500, 1000, 5000], on each chosen candidate's *training split* only. Scoping only, to inform a vocabulary size choice later -- no capping is applied here.

In [ ]:
K_VALUES = [50, 100, 250, 500, 1000, 2000, 5000]
candidates_for_vocab = {
    "tmdb_budget_revenue_gt0": tmdb_br_gt0_train,
    "tmdb_no_budget_revenue_gt5": tmdb_nbr_gt5_train,
}

def topk_coverage(df, feature, k_values=K_VALUES):
    freq = df[feature].explode().value_counts()
    total = freq.sum()
    return pd.Series({k: freq.head(k).sum() / total for k in k_values})

### `keywords` top-K coverage

In [ ]:
keywords_topk_coverage = pd.DataFrame(
    {name: topk_coverage(df, "keywords") for name, df in candidates_for_vocab.items()}
)
keywords_topk_coverage.index.name = "K"
keywords_topk_coverage.round(3)

### `production_companies` top-K coverage

In [ ]:
production_companies_topk_coverage = pd.DataFrame(
    {name: topk_coverage(df, "production_companies") for name, df in candidates_for_vocab.items()}
)
production_companies_topk_coverage.index.name = "K"
production_companies_topk_coverage.round(3)

### Coverage & marginal gain curves

For each candidate's training split and feature, plots the exact cumulative coverage curve (every K from 1 to 10000) alongside a smoothed marginal-gain curve (100-item rolling window) to visually spot where returns diminish. Still diagnostic only -- no vocabulary is capped here.

In [ ]:
def find_elbow(coverage):
    # Elbow = the K whose (min-max normalized) point on the coverage curve
    # is furthest (perpendicular distance) from the straight line connecting
    # the curve's first and last points -- the classic "kneedle" heuristic.
    k_values = np.arange(1, len(coverage) + 1)
    k_norm = k_values / k_values.max()
    cov_norm = coverage.values / coverage.values.max()

    x1, y1 = k_norm[0], cov_norm[0]
    x2, y2 = k_norm[-1], cov_norm[-1]

    numerator = np.abs((y2 - y1) * k_norm - (x2 - x1) * cov_norm + x2 * y1 - y2 * x1)
    denominator = np.sqrt((y2 - y1) ** 2 + (x2 - x1) ** 2)
    distances = numerator / denominator

    elbow_idx = np.argmax(distances)
    return k_values[elbow_idx], coverage.values[elbow_idx]

In [ ]:
# Muted blue/orange pair (not matplotlib's saturated named "blue"/"red") and
# near-black ink, closer to a publication figure than a default textbook plot.
CURVE_COLOR = "#2a78d6"
ELBOW_COLOR = "#eb6834"
TEXT_COLOR = "#0b0b0b"

features = ["keywords", "production_companies"]

for candidate_name, df in candidates_for_vocab.items():
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

    for ax, feature in zip(axes, features):
        freq = df[feature].explode().value_counts()
        coverage = freq.head(10000).cumsum() / freq.sum()
        k_values = range(1, len(coverage) + 1)

        elbow_k, elbow_coverage = find_elbow(coverage)
        print(f"{candidate_name} / {feature}: elbow at K={elbow_k}, coverage={elbow_coverage:.1%}")

        ax.plot(k_values, coverage.values, color=CURVE_COLOR, linewidth=2)
        ax.axvline(elbow_k, color=ELBOW_COLOR, linestyle="--", linewidth=1.5)
        ax.annotate(
            f"K = {elbow_k}\n{elbow_coverage:.1%} coverage",
            xy=(elbow_k, elbow_coverage),
            xytext=(15, -25),
            textcoords="offset points",
            fontsize=11,
            color=ELBOW_COLOR,
        )

        ax.set_title(feature, fontsize=13, color=TEXT_COLOR)
        ax.set_xlabel("K (top-K items by frequency)", fontsize=12, color=TEXT_COLOR)
        ax.set_ylabel("Cumulative coverage", fontsize=12, color=TEXT_COLOR)
        ax.set_xticks(range(0, 10001, 2500))
        ax.tick_params(axis="both", labelsize=10, colors=TEXT_COLOR)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.suptitle(candidate_name, fontsize=14, color=TEXT_COLOR)
    fig.tight_layout()
    savefig(fig, f"03_{candidate_name}_vocab_coverage_elbows.png")
    plt.show()

### Apply the vocabulary cap

Fixes K = 2000 for both features. Kept-item sets and coverage are computed on the training split only (via the same `explode().value_counts()` used above); val and test reuse that exact set and K instead of recomputing their own frequencies, and every item outside it becomes `"Other"` in all three splits.

In [ ]:
VOCAB_TOP_K = 2000
list_features = ["keywords", "production_companies"]

def bucket_rare_items(train_df, val_df, test_df, feature, top_k=VOCAB_TOP_K):
    freq = train_df[feature].explode().value_counts()
    kept_items = set(freq.head(top_k).index)
    train_coverage = freq.head(top_k).sum() / freq.sum()

    def bucket_list(items):
        # pd.read_parquet round-trips list cells into numpy arrays, not
        # Python lists, so isinstance(items, list) alone would silently
        # skip every row.
        if not pd.api.types.is_list_like(items):
            return items
        result = []
        seen_other = False
        for item in items:
            if item in kept_items:
                result.append(item)
            elif not seen_other:
                result.append("Other")
                seen_other = True
        return result

    def apply_bucket(df):
        df = df.copy()
        df[feature] = df[feature].apply(bucket_list)
        return df

    return apply_bucket(train_df), apply_bucket(val_df), apply_bucket(test_df), kept_items, train_coverage

In [ ]:
tmdb_br_gt0_train, tmdb_br_gt0_val, tmdb_br_gt0_test, br_gt0_keywords_vocab, br_gt0_keywords_coverage = bucket_rare_items(
    tmdb_br_gt0_train, tmdb_br_gt0_val, tmdb_br_gt0_test, "keywords"
)
tmdb_br_gt0_train, tmdb_br_gt0_val, tmdb_br_gt0_test, br_gt0_companies_vocab, br_gt0_companies_coverage = bucket_rare_items(
    tmdb_br_gt0_train, tmdb_br_gt0_val, tmdb_br_gt0_test, "production_companies"
)

tmdb_nbr_gt5_train, tmdb_nbr_gt5_val, tmdb_nbr_gt5_test, nbr_gt5_keywords_vocab, nbr_gt5_keywords_coverage = bucket_rare_items(
    tmdb_nbr_gt5_train, tmdb_nbr_gt5_val, tmdb_nbr_gt5_test, "keywords"
)
tmdb_nbr_gt5_train, tmdb_nbr_gt5_val, tmdb_nbr_gt5_test, nbr_gt5_companies_vocab, nbr_gt5_companies_coverage = bucket_rare_items(
    tmdb_nbr_gt5_train, tmdb_nbr_gt5_val, tmdb_nbr_gt5_test, "production_companies"
)

print(f"tmdb_budget_revenue_gt0 / keywords: vocab_size={len(br_gt0_keywords_vocab) + 1}, train coverage={br_gt0_keywords_coverage:.1%}")
print(f"tmdb_budget_revenue_gt0 / production_companies: vocab_size={len(br_gt0_companies_vocab) + 1}, train coverage={br_gt0_companies_coverage:.1%}")
print(f"tmdb_no_budget_revenue_gt5 / keywords: vocab_size={len(nbr_gt5_keywords_vocab) + 1}, train coverage={nbr_gt5_keywords_coverage:.1%}")
print(f"tmdb_no_budget_revenue_gt5 / production_companies: vocab_size={len(nbr_gt5_companies_vocab) + 1}, train coverage={nbr_gt5_companies_coverage:.1%}")

## 6. Missingness check

For each chosen candidate, one table with `train`/`val`/`test` as columns and features as rows, each cell showing the null count alongside the null percentage for that split. Should come back clean, or close to it, now that missing `runtime` rows were dropped in `01_overview_cleaning.ipynb` and missing `overview` values were filled with an empty string in `02_preprocessing.ipynb`.

In [ ]:
def missingness_summary(train_df, val_df, test_df):
    summary = {}
    for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        counts = df.isnull().sum()
        pct = df.isnull().mean() * 100
        summary[split_name] = counts.astype(str) + " (" + pct.round(2).astype(str) + "%)"
    return pd.DataFrame(summary)


br_gt0_missingness = missingness_summary(tmdb_br_gt0_train, tmdb_br_gt0_val, tmdb_br_gt0_test)
nbr_gt5_missingness = missingness_summary(tmdb_nbr_gt5_train, tmdb_nbr_gt5_val, tmdb_nbr_gt5_test)

missingness_after_split = {
    "tmdb_br_gt0_train": int(tmdb_br_gt0_train.isnull().sum().sum()),
    "tmdb_br_gt0_val": int(tmdb_br_gt0_val.isnull().sum().sum()),
    "tmdb_br_gt0_test": int(tmdb_br_gt0_test.isnull().sum().sum()),
    "tmdb_nbr_gt5_train": int(tmdb_nbr_gt5_train.isnull().sum().sum()),
    "tmdb_nbr_gt5_val": int(tmdb_nbr_gt5_val.isnull().sum().sum()),
    "tmdb_nbr_gt5_test": int(tmdb_nbr_gt5_test.isnull().sum().sum()),
}

print("tmdb_budget_revenue_gt0:")
display(br_gt0_missingness)

print("tmdb_no_budget_revenue_gt5:")
display(nbr_gt5_missingness)

## 7. Restore `original_language` category dtype

Section 3's `bucket_rare_languages` had to cast `original_language` to plain `str` via `.astype(str).where(mask, "other")`, since pandas' `Categorical` can't be assigned a value (`"other"`) outside its fixed category list. That cast is what's still in effect on all six splits at this point. Restores `category` dtype here, right before saving, so the files in `data/final/` carry the same dtype `original_language` had everywhere upstream.

In [ ]:
final_splits_for_dtype_fix = [
    tmdb_br_gt0_train, tmdb_br_gt0_val, tmdb_br_gt0_test,
    tmdb_nbr_gt5_train, tmdb_nbr_gt5_val, tmdb_nbr_gt5_test,
]

for df in final_splits_for_dtype_fix:
    df["original_language"] = df["original_language"].astype("category")

print("original_language dtype:", tmdb_br_gt0_train["original_language"].dtype)


## 8. Save final splits

Saves all six train/val/test splits, for both chosen candidates, to `data/final/`.

In [ ]:
FINAL_DIR = REPO_ROOT / "data" / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

final_splits = {
    "tmdb_br_gt0_train": tmdb_br_gt0_train,
    "tmdb_br_gt0_val": tmdb_br_gt0_val,
    "tmdb_br_gt0_test": tmdb_br_gt0_test,
    "tmdb_nbr_gt5_train": tmdb_nbr_gt5_train,
    "tmdb_nbr_gt5_val": tmdb_nbr_gt5_val,
    "tmdb_nbr_gt5_test": tmdb_nbr_gt5_test,
}

final_paths = {}
for name, df in final_splits.items():
    path = FINAL_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    final_paths[name] = path
    print(f"Saved: {path} ({df.shape})")

## 9. Log summary

In [ ]:
findings = {
    "Split sizes (80/10/10)": {
        "train": tmdb_br_gt0_train.shape,
        "val": tmdb_br_gt0_val.shape,
        "test": tmdb_br_gt0_test.shape,
    },
    "Split sizes (70/15/15)": {
        "train": tmdb_nbr_gt5_train.shape,
        "val": tmdb_nbr_gt5_val.shape,
        "test": tmdb_nbr_gt5_test.shape,
    },
    "Stratified on": "original_language, via a disposable stratify_key that pools languages under 10 total occurrences into 'other' (see Section 1)",
    "Rare-language bucketing (train-derived, min_count=100)": {
        "tmdb_budget_revenue_gt0": f"{len(br_gt0_kept_languages)} kept languages + 'other'",
        "tmdb_no_budget_revenue_gt5": f"{len(nbr_gt5_kept_languages)} kept languages + 'other'",
    },
    "original_language embedding dim (raw 179 -> post-binning)": {
        "raw (00_tmdb_eda.ipynb)": raw_embedding_dim,
        "tmdb_budget_revenue_gt0": int(embedding_dim_comparison.loc["tmdb_budget_revenue_gt0", "suggested_embedding_dim"]),
        "tmdb_no_budget_revenue_gt5": int(embedding_dim_comparison.loc["tmdb_no_budget_revenue_gt5", "suggested_embedding_dim"]),
    },
    "Vocabulary cap (train-derived top 2000 + 'Other')": {
        "tmdb_budget_revenue_gt0 / keywords": f"vocab_size={len(br_gt0_keywords_vocab) + 1}, train coverage={br_gt0_keywords_coverage:.1%}",
        "tmdb_budget_revenue_gt0 / production_companies": f"vocab_size={len(br_gt0_companies_vocab) + 1}, train coverage={br_gt0_companies_coverage:.1%}",
        "tmdb_no_budget_revenue_gt5 / keywords": f"vocab_size={len(nbr_gt5_keywords_vocab) + 1}, train coverage={nbr_gt5_keywords_coverage:.1%}",
        "tmdb_no_budget_revenue_gt5 / production_companies": f"vocab_size={len(nbr_gt5_companies_vocab) + 1}, train coverage={nbr_gt5_companies_coverage:.1%}",
    },
    "Missing values remaining per split (total cells)": missingness_after_split,
    "Saved to": {name: str(path) for name, path in final_paths.items()},
}

log_findings(
    title="03 Feature Split: TMDB train/val/test -- summary",
    findings=findings,
    figures=[
        "reports/figures/03_tmdb_budget_revenue_gt0_language_coverage.png",
        "reports/figures/03_tmdb_no_budget_revenue_gt5_language_coverage.png",
        "reports/figures/03_tmdb_budget_revenue_gt0_vocab_coverage_elbows.png",
        "reports/figures/03_tmdb_no_budget_revenue_gt5_vocab_coverage_elbows.png",
    ],
)
print("Logged summary to reports/eda_log.md")